#### THIRD ATTEMPT

## Data Exploration

### OpTc Dataset
Overview
The OpTC (Operationally Transparent Cyber) dataset was developed by Five Directions, under the DARPA Transparent Computing programme, to support research into large-scale cyber-security monitoring and attack detection. It contains endpoint telemetry collected from Windows 10 computers, recording system and network activity through eCAR (extended Cyber Analytics Repository) events. This dataset contains records, including things such as processes, files, network flows, registry activity and other host events. The original release contains roughly a terabyte of compressed data from hundreds of hosts. It includes benign activity as well as red-team attack activity.

I will be using the corrected 2026 version of the OpTC dataset. INRIA reports that the original dataset contains errors involving unique identifiers and other event properties and recommends using the corrected version instead. It is available at: https://entrepot.recherche.data.gouv.fr/dataset.xhtml?persistentId=doi%3A10.57745%2FUXCWOC&utm_source=chatgpt.com

**project goal:**

The goal of this project is to investigate the use of ML/DL to predict the occurrence of cybersecurity attacks. The project will first analyse and preprocess the OpTC event data to identify behavioural patterns associated with malicious activity, before transforming the sequential telemetry into suitable features and time-based samples for modelling. Different ML/DL approaches will then be evaluated to determine whether patterns in system and network activity can provide sufficient information to identify or predict an impending cyber attack.


In [1]:
# Imports
import json
import pandas as pd
from pathlib import Path



#### 1. Load all files into an array

In [2]:

# Get all the processed Parquet files
files = list(Path("../data/processed").glob("*.parquet"))

# Check if all were loaded correctly
print(f"Number of files: {len(files)}")


Number of files: 50


#### 2. Start EDA

In [3]:
# I am checking the first processed file to make sure everything looks correct
data = pd.read_parquet(files[0])

print(data.shape)
data.head()

(2358415, 13)


,action,actorID,hostname,id,object,objectID,pid,ppid,principal,properties,tid,timestamp,date
0,START,824b13ee-6b32-4fbf-9321-4f1945a5c83a,SysClient0056.systemia.com,7597ec82-0960-49ca-9e25-db0b87e556ec,FLOW,e6ce56f2-30b1-4b2e-8f96-e6f6b19f851f,2140,1836,NT AUTHORITY\SYSTEM,"{'acuity_level': '1', 'base_address': None, 'c...",-1,2019-09-25 00:00:00.061000-04:00,2019-09-25
1,OPEN,365d2928-bcab-4356-9385-9edb28a103f5,SysClient0056.systemia.com,f388cef5-61fc-4553-9edf-4aa01f4a1989,PROCESS,307eeab4-3de9-4e6f-af78-baced7bf2e91,664,560,NT AUTHORITY\SYSTEM,"{'acuity_level': '5', 'base_address': None, 'c...",3396,2019-09-25 00:00:00.062000-04:00,2019-09-25
2,OPEN,365d2928-bcab-4356-9385-9edb28a103f5,SysClient0056.systemia.com,a9f182f7-5cd4-41e7-afa2-118f58fb8644,PROCESS,307eeab4-3de9-4e6f-af78-baced7bf2e91,664,560,NT AUTHORITY\SYSTEM,"{'acuity_level': '5', 'base_address': None, 'c...",3396,2019-09-25 00:00:00.062000-04:00,2019-09-25
3,REMOTE_CREATE,307eeab4-3de9-4e6f-af78-baced7bf2e91,SysClient0056.systemia.com,b447339c-5701-4e78-af0e-285f881abf3f,THREAD,9a69d2d3-d7ea-46da-884e-dfca13f83640,344,560,NT AUTHORITY\SYSTEM,"{'acuity_level': '3', 'base_address': None, 'c...",472,2019-09-25 00:00:00.065000-04:00,2019-09-25
4,CREATE,307eeab4-3de9-4e6f-af78-baced7bf2e91,SysClient0056.systemia.com,aab28347-bf2f-42e8-a92e-7ec44d124804,PROCESS,ee439f5f-2ee3-47f4-9553-41e41575aba6,4360,344,NT AUTHORITY\SYSTEM,"{'acuity_level': '1', 'base_address': None, 'c...",-1,2019-09-25 00:00:00.065000-04:00,2019-09-25


In [4]:
# I am checking the structure of the data
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2358415 entries, 0 to 2358414
Data columns (total 13 columns):
 #   Column      Dtype                                 
---  ------      -----                                 
 0   action      object                                
 1   actorID     object                                
 2   hostname    object                                
 3   id          object                                
 4   object      object                                
 5   objectID    object                                
 6   pid         int64                                 
 7   ppid        int64                                 
 8   principal   object                                
 9   properties  object                                
 10  tid         int64                                 
 11  timestamp   datetime64[ns, pytz.FixedOffset(-240)]
 12  date        object                                
dtypes: datetime64[ns, pytz.FixedOffset(-240)](

In [5]:
# data["object"].value_counts()
# data["action"].value_counts()

# # I am checking the properties available for each type of event
# for event_type in data["object"].unique():
#     print(f"\n{event_type}")
    
#     sample = data[data["object"] == event_type]["properties"].dropna().iloc[0]
#     print(sample)



In [6]:
# Expand the properties column

properties_df = pd.json_normalize(data["properties"])

data = pd.concat(
    [data.drop(columns=["properties"]), properties_df],
    axis=1
)

# confirm that it is done
data.head()
data.shape

(2358415, 59)

In [7]:
print(data.columns.tolist())
print(data.dtypes)

['action', 'actorID', 'hostname', 'id', 'object', 'objectID', 'pid', 'ppid', 'principal', 'tid', 'timestamp', 'date', 'acuity_level', 'base_address', 'command_line', 'context_info', 'data', 'dest_ip', 'dest_port', 'direction', 'end_time', 'file_path', 'image_path', 'info_class', 'key', 'l4protocol', 'logon_id', 'module_path', 'new_path', 'parent_image_path', 'path', 'payload', 'privileges', 'requesting_domain', 'requesting_logon_id', 'requesting_user', 'sid', 'size', 'src_ip', 'src_pid', 'src_port', 'src_tid', 'stack_base', 'stack_limit', 'start_address', 'start_time', 'subprocess_tag', 'task_name', 'task_pid', 'task_process_uuid', 'tgt_pid', 'tgt_pid_uuid', 'tgt_tid', 'type', 'user', 'user_name', 'user_stack_base', 'user_stack_limit', 'value']
action                                                 object
actorID                                                object
hostname                                               object
id                                                     obje

In [8]:
print(data["object"].value_counts())
print(data["action"].value_counts())
print(data["date"].value_counts())

object
FLOW            1886707
FILE             226573
PROCESS           91503
MODULE            86165
THREAD            59848
REGISTRY           6290
TASK               1126
USER_SESSION        195
SHELL                 7
HOST                  1
Name: count, dtype: int64
action
START            1364663
MESSAGE           521238
OPEN               88032
LOAD               86165
MODIFY             83203
WRITE              61626
CREATE             50568
READ               44518
TERMINATE          32803
RENAME             10794
REMOTE_CREATE       5204
EDIT                3165
DELETE              3109
ADD                 1694
REMOVE              1431
GRANT                 89
LOGIN                 74
REMOTE                14
LOGOUT                14
COMMAND                7
INTERACTIVE            4
Name: count, dtype: int64
date
2019-09-25    2358415
Name: count, dtype: int64


In [9]:
data['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

print(data['timestamp'].min())
print(data['timestamp'].max())
print(data['timestamp'].isna().sum())

NameError: name 'df' is not defined

In [ ]:
print(data["acuity_level"].value_counts(dropna=False))
print(data.isna().mean().sort_values(ascending=False))

acuity_level
1    1366521
4     527962
3     190469
5     185373
2      86262
0       1828
Name: count, dtype: int64
context_info           0.999997
payload                0.999997
requesting_domain      0.999967
requesting_user        0.999967
privileges             0.999962
requesting_logon_id    0.999961
user_name              0.999953
logon_id               0.999917
task_process_uuid      0.999570
path                   0.999570
task_pid               0.999570
task_name              0.999523
value                  0.998294
type                   0.998033
data                   0.997810
tgt_pid_uuid           0.997793
sid                    0.997756
user                   0.997673
key                    0.997333
new_path               0.995423
end_time               0.995167
start_time             0.995167
src_tid                0.974624
subprocess_tag         0.974624
start_address          0.974624
stack_limit            0.974624
tgt_pid                0.974624
user_stack_base    